# Tutorial 19: HLLSet AI — The Complete Pipeline

## From Documents to Intelligent Retrieval in 6 Lines

This tutorial demonstrates the **complete HLLSet AI pipeline**:

```python
from core import HLLSetTransformer, HLLLattice

transformer = HLLSetTransformer(lattice=lattice)
result = transformer.forward("your query here")
doc_graph = result.disambiguate(engine, registry)
response = doc_graph.to_llm_prompt()  # or send to external LLM
```

**That's it.** Six lines from query to structured response.

---

## What This Tutorial Covers

1. **Document Ingestion** — Build a multi-layered W lattice from document batches
2. **Lattice Inspection** — Examine the temporal structure
3. **Query Processing** — Run the HLLSet Transformer
4. **Graph Disambiguation** — Recover documents with τ-edge relationships
5. **Response Synthesis** — Simulate LLM handoff (concatenation)
6. **Re-ingestion** — Close the loop by adding synthesis to the lattice

---

## The Architecture

```
Documents → Ingest → W Lattice (multi-layer)
                         ↓
Query → HLLSetTransformer → Complement-based attention
                         ↓
        HLLSet Graph (with τ edges)
                         ↓
        Disambiguate → Document Graph
                         ↓
        LLM Synthesis (or simulated)
                         ↓
        Re-ingest → W Lattice grows!
```

## Setup

In [1]:
import sys
sys.path.insert(0, '..')

# Force reload to pick up transformer changes
import importlib
import core.hllset_transformer
importlib.reload(core.hllset_transformer)

from core import (
    HLLSet,
    HLLLattice,
    DisambiguationEngine,
    bss,
    compute_sha1,
)
from core.hllset_transformer import HLLSetTransformer, TransformerResult

print("✓ All modules imported successfully")
print(f"\nHLLSet AI Pipeline ready!")

✓ All modules imported successfully

HLLSet AI Pipeline ready!


---

## Part 1: Document Corpus

We'll create a small corpus about **world capitals** — a classic knowledge retrieval task.

The documents will be ingested in **multiple batches** to create a multi-layered W lattice,
simulating how a real system accumulates knowledge over time.

In [2]:
# Document corpus organized by topic and time of ingestion

# Batch 1: European capitals (t=1.0)
BATCH_1 = [
    "Paris is the capital of France. It is known for the Eiffel Tower and the Louvre museum.",
    "London is the capital of the United Kingdom. Big Ben and Buckingham Palace are famous landmarks.",
    "Berlin is the capital of Germany. The Brandenburg Gate is a symbol of reunification.",
    "Rome is the capital of Italy. The Colosseum and Vatican City attract millions of visitors.",
    "Madrid is the capital of Spain. The Prado Museum houses masterpieces by Goya and Velázquez.",
]

# Batch 2: Asian capitals (t=2.0)
BATCH_2 = [
    "Tokyo is the capital of Japan. It is a megacity blending ancient temples with modern skyscrapers.",
    "Beijing is the capital of China. The Forbidden City and Great Wall are UNESCO World Heritage sites.",
    "New Delhi is the capital of India. It houses the Parliament and India Gate memorial.",
    "Seoul is the capital of South Korea. Gyeongbokgung Palace represents traditional Korean architecture.",
    "Bangkok is the capital of Thailand. The Grand Palace and Wat Phra Kaew are sacred landmarks.",
]

# Batch 3: Americas capitals (t=3.0)
BATCH_3 = [
    "Washington DC is the capital of the United States. The White House and Capitol Building are iconic.",
    "Ottawa is the capital of Canada. Parliament Hill overlooks the Ottawa River.",
    "Mexico City is the capital of Mexico. The Zócalo is one of the largest public squares in the world.",
    "Brasília is the capital of Brazil. It was designed by Oscar Niemeyer with modernist architecture.",
    "Buenos Aires is the capital of Argentina. La Boca neighborhood is famous for colorful houses and tango.",
]

# Batch 4: Additional context about France (t=4.0)
BATCH_4 = [
    "France is a country in Western Europe. French cuisine is renowned worldwide for its sophistication.",
    "The French Revolution began in Paris in 1789. It transformed France from monarchy to republic.",
    "Paris hosts the 2024 Summer Olympics. Events will take place at iconic venues including the Eiffel Tower.",
    "The Louvre in Paris contains over 35000 artworks including the Mona Lisa by Leonardo da Vinci.",
]

ALL_BATCHES = [
    (1.0, "European capitals", BATCH_1),
    (2.0, "Asian capitals", BATCH_2),
    (3.0, "Americas capitals", BATCH_3),
    (4.0, "France context", BATCH_4),
]

print(f"Corpus prepared:")
for t, name, batch in ALL_BATCHES:
    print(f"  t={t}: {name} ({len(batch)} documents)")
print(f"\nTotal: {sum(len(b) for _, _, b in ALL_BATCHES)} documents")

Corpus prepared:
  t=1.0: European capitals (5 documents)
  t=2.0: Asian capitals (5 documents)
  t=3.0: Americas capitals (5 documents)
  t=4.0: France context (4 documents)

Total: 19 documents


---

## Part 2: Build the W Lattice

We ingest documents in batches, creating a **multi-layered temporal lattice**.

Each layer has:
- A **merged HLLSet** (union of all documents at that time)
- A **level ID** = SHA1(merged) — content-addressed identity
- **Individual HLLSets** for each document

In [3]:
# Create lattice and disambiguation engine
P_BITS = 10

lattice = HLLLattice(p_bits=P_BITS)
engine = DisambiguationEngine(p_bits=P_BITS)

print("Ingesting documents into W lattice...\n")

for timestamp, batch_name, documents in ALL_BATCHES:
    # Create HLLSets for each document
    doc_hllsets = []
    
    for doc in documents:
        tokens = doc.lower().split()
        
        # Create HLLSet for document
        hll = HLLSet.from_batch(tokens, p_bits=P_BITS)
        doc_hllsets.append(hll)
        
        # Register n-grams for later disambiguation
        engine.ingest_tokens(tokens)
    
    # Append to lattice as a single node (merged)
    node = lattice.append(
        doc_hllsets,
        timestamp=timestamp,
        metadata={'batch_name': batch_name, 'doc_count': len(documents)}
    )
    
    print(f"t={timestamp}: {batch_name}")
    print(f"  Level ID: {node.node_id[:16]}...")
    print(f"  Cardinality: {node.cardinality:.0f} unique tokens")
    print(f"  Documents: {len(documents)}")
    print()

print(f"\n✓ W Lattice built with {lattice.node_count} levels")

Ingesting documents into W lattice...

t=1.0: European capitals
  Level ID: 5db5fd678b91aa82...
  Cardinality: 47 unique tokens
  Documents: 5

t=2.0: Asian capitals
  Level ID: 5c0fa87d9427cb66...
  Cardinality: 53 unique tokens
  Documents: 5

t=3.0: Americas capitals
  Level ID: 189fca609cd482a4...
  Cardinality: 57 unique tokens
  Documents: 5

t=4.0: France context
  Level ID: 87aec656f7fe92c1...
  Cardinality: 55 unique tokens
  Documents: 4


✓ W Lattice built with 4 levels


### Inspect Lattice Structure

In [4]:
# Show cumulative growth over time
print("Cumulative knowledge at each level:")
print("=" * 50)

for node in lattice.all_nodes():
    cumulative = lattice.cumulative(t=node.timestamp)
    print(f"t={node.timestamp}: |M(t)|≈{cumulative.cardinality():.0f} tokens (level: {node.node_id[:12]}...)")

print("\nFull corpus coverage:")
full_cumulative = lattice.cumulative()
print(f"  Total unique tokens: {full_cumulative.cardinality():.0f}")

Cumulative knowledge at each level:
t=1.0: |M(t)|≈47 tokens (level: 5db5fd678b91...)
t=2.0: |M(t)|≈88 tokens (level: 5c0fa87d9427...)
t=3.0: |M(t)|≈129 tokens (level: 189fca609cd4...)
t=4.0: |M(t)|≈170 tokens (level: 87aec656f7fe...)

Full corpus coverage:
  Total unique tokens: 170


In [5]:
# Engine already created above - verify it
print(f"DisambiguationEngine ready:")
print(f"  Ingested n-grams from {sum(len(b) for _, _, b in ALL_BATCHES)} documents")
print(f"  Can recover tokens from any HLLSet in the corpus")

DisambiguationEngine ready:
  Ingested n-grams from 19 documents
  Can recover tokens from any HLLSet in the corpus


---

## Part 3: The HLLSet AI Model — 6 Lines of Magic

Here it is — the complete HLLSet AI model:

```python
transformer = HLLSetTransformer(lattice=lattice)
result = transformer.forward("what is the capital of france")
doc_graph = result.disambiguate(engine, registry)
response = synthesize(doc_graph)  # LLM or simulation
lattice.append([HLLSet.from_batch(response.split())])  # re-ingest
```

Let's run it!

In [6]:
# THE HLLSET AI MODEL
# ===================

# Create transformer with temperature and BSS threshold controls
transformer = HLLSetTransformer(
    lattice=lattice,
    p_bits=P_BITS,
    
    # BSS Thresholds (like attention weights)
    tau_threshold=0.05,      # Minimum τ (coverage) for relevance
    rho_threshold=0.8,       # Maximum ρ (noise) to tolerate
    
    # Temperature Control (like LLM temperature)
    # 0.0 = strict/focused, 0.5 = balanced, 1.0 = exploratory/creative
    temperature=0.5,
    
    max_depth=10,            # Maximum backward propagation
    convergence_tolerance=0.05,
)

print("✓ HLLSetTransformer created")
print(f"  Lattice levels: {lattice.node_count}")
print(f"  Max depth: {transformer._max_depth}")
print(f"  Temperature: {transformer.temperature}")
print(f"  τ threshold: {transformer.tau_threshold}")
print(f"  ρ threshold: {transformer.rho_threshold}")

✓ HLLSetTransformer created
  Lattice levels: 4
  Max depth: 10
  Temperature: 0.5
  τ threshold: 0.05
  ρ threshold: 0.8


### Query 1: "What is the capital of France?"

In [7]:
# Run the query
QUERY_1 = "what is the capital of france"

print(f"Query: '{QUERY_1}'")
print("=" * 60)

# Forward pass through the transformer
result_1 = transformer.forward(QUERY_1)

print(f"\n{result_1.summary()}")
print(f"\nResult ID: {result_1.result_id[:16]}...")
print(f"Context cardinality: {result_1.final_context.cardinality():.0f} tokens")
print(f"HLLSet graph: {result_1.hllset_graph}")

Query: 'what is the capital of france'

Transformer: 4 levels, +168 info, lattice_boundary

Result ID: 6496e6fcd74923d1...
Context cardinality: 172 tokens
HLLSet graph: HLLSetGraph(4 nodes, 0 edges)


### Examine the Attention Trace

In [8]:
print("Attention trace (backward propagation through time):")
print("=" * 70)

for rec in result_1.attention_trace:
    status = "EXHAUSTED" if rec.is_exhausted else f"τ={rec.tau_to_query:.3f}, ρ={rec.rho_noise:.3f}"
    noisy = " [NOISY]" if rec.is_noisy else ""
    print(f"  Depth {rec.depth}: level={rec.level_id[:8]}... | "
          f"|Δ|≈{rec.complement_cardinality:.0f} | "
          f"{status}{noisy} | "
          f"Φ={rec.information_gain:.0f}")

print(f"\nNoether flux history: {result_1.noether_flux_history}")
print(f"Convergence: {result_1.convergence_reason.value}")

Attention trace (backward propagation through time):
  Depth 0: level=87aec656... | |Δ|≈53 | τ=0.571, ρ=0.927 [NOISY] | Φ=53
  Depth 1: level=189fca60... | |Δ|≈49 | τ=0.714, ρ=0.912 [NOISY] | Φ=49
  Depth 2: level=5c0fa87d... | |Δ|≈39 | τ=0.714, ρ=0.906 [NOISY] | Φ=39
  Depth 3: level=5db5fd67... | |Δ|≈27 | τ=0.714, ρ=0.894 [NOISY] | Φ=27

Noether flux history: (53, 49, 39, 27)
Convergence: lattice_boundary


### Temperature Effects

Let's see how temperature affects retrieval:

In [9]:
# Compare different temperature settings
print("Temperature Comparison for same query:")
print("=" * 60)

query = "capital of france paris"

for temp in [0.0, 0.5, 1.0]:
    transformer.temperature = temp
    result = transformer.forward(query)
    print(f"\nTemperature {temp}:")
    print(f"  Info gain: +{result.total_information_gain:.0f}")
    print(f"  Context size: {result.final_context.cardinality():.0f} tokens")
    print(f"  HLLSet nodes: {result.hllset_graph.node_count}")
    print(f"  Convergence: {result.convergence_reason.value}")

# Reset to default
transformer.temperature = 0.5
print(f"\n(Reset temperature to {transformer.temperature})")

Temperature Comparison for same query:

Temperature 0.0:
  Info gain: +124
  Context size: 130 tokens
  HLLSet nodes: 3
  Convergence: lattice_boundary

Temperature 0.5:
  Info gain: +169
  Context size: 170 tokens
  HLLSet nodes: 4
  Convergence: lattice_boundary

Temperature 1.0:
  Info gain: +169
  Context size: 170 tokens
  HLLSet nodes: 4
  Convergence: lattice_boundary

(Reset temperature to 0.5)


### W-Markov Chain Approach

The Markov Chain view of the W lattice:
- Each HLLSet is a **state** in the chain
- **Transition probability** ∝ τ(current → candidate)
- We **collect state IDs** (not merge into blob)
- Merge only for Noether check

Key difference: preserves **individual document provenance**.

In [10]:
# Test the W-Markov Chain forward pass
print("W-Markov Chain Forward Pass:")
print("=" * 60)

# Use forward_mc instead of forward
result_mc = transformer.forward_mc(QUERY_1)

print(f"\nQuery: '{QUERY_1}'")
print(f"\n{result_mc.summary()}")
print(f"\nCollected HLLSets: {result_mc.hllset_graph.node_count}")
print(f"Graph edges (transitions): {result_mc.hllset_graph.edge_count}")
print(f"Final context: {result_mc.final_context.cardinality():.0f} tokens")

# Show the preserved provenance
print("\n--- Preserved Provenance (individual HLLSet IDs) ---")
for node_id, hll in result_mc.hllset_graph.nodes.items():
    print(f"  {node_id[:12]}... → |M|≈{hll.cardinality():.0f} tokens")

# Show transitions (Markov edges)
if result_mc.hllset_graph.edges:
    print("\n--- Markov Transitions ---")
    for edge in result_mc.hllset_graph.edges:
        print(f"  {edge.source_id[:8]}→{edge.target_id[:8]} (τ={edge.tau:.3f})")

W-Markov Chain Forward Pass:

Query: 'what is the capital of france'

Transformer: 4 levels, +165 info, lattice_boundary

Collected HLLSets: 5
Graph edges (transitions): 4
Final context: 172 tokens

--- Preserved Provenance (individual HLLSet IDs) ---
  49cb69b90973... → |M|≈7 tokens
  87aec656f7fe... → |M|≈55 tokens
  189fca609cd4... → |M|≈57 tokens
  5c0fa87d9427... → |M|≈53 tokens
  5db5fd678b91... → |M|≈47 tokens

--- Markov Transitions ---
  49cb69b9→87aec656 (τ=0.571)
  87aec656→189fca60 (τ=0.714)
  189fca60→5c0fa87d (τ=0.714)
  5c0fa87d→5db5fd67 (τ=0.714)


In [11]:
# Compare: Original forward vs Markov Chain forward
print("COMPARISON: forward() vs forward_mc()")
print("=" * 60)

result_orig = transformer.forward(QUERY_1)
result_mc = transformer.forward_mc(QUERY_1)

print(f"\n{'Metric':<30} {'forward()':<20} {'forward_mc()':<20}")
print("-" * 70)
print(f"{'Final context tokens':<30} {result_orig.final_context.cardinality():<20.0f} {result_mc.final_context.cardinality():<20.0f}")
print(f"{'HLLSet nodes (graph)':<30} {result_orig.hllset_graph.node_count:<20} {result_mc.hllset_graph.node_count:<20}")
print(f"{'Graph edges':<30} {result_orig.hllset_graph.edge_count:<20} {result_mc.hllset_graph.edge_count:<20}")
print(f"{'Information gain':<30} {result_orig.total_information_gain:<20.0f} {result_mc.total_information_gain:<20.0f}")
print(f"{'Convergence':<30} {result_orig.convergence_reason.value:<20} {result_mc.convergence_reason.value:<20}")

print("\n✓ forward_mc() preserves individual HLLSet provenance!")
print("  Each collected HLLSet has its own SHA1 ID")
print("  Markov transitions show the retrieval path")

COMPARISON: forward() vs forward_mc()

Metric                         forward()            forward_mc()        
----------------------------------------------------------------------
Final context tokens           172                  172                 
HLLSet nodes (graph)           4                    5                   
Graph edges                    0                    4                   
Information gain               168                  165                 
Convergence                    lattice_boundary     lattice_boundary    

✓ forward_mc() preserves individual HLLSet provenance!
  Each collected HLLSet has its own SHA1 ID
  Markov transitions show the retrieval path


### 2-Phase Architecture (NEW!)

The **improved architecture** separates concerns:

1. **Phase 1: Collection** (`collect_context()`)
   - Uses **lattice topology** to find relevant HLLSets
   - Returns `CollectedContext` with individual HLLSets + provenance
   - Stop criterion: Noether convergence, boundary, or max depth

2. **Phase 2: Markov Chain** (`build_markov_chain()`)  
   - Uses **BSS** to compute semantic transition probabilities
   - P(j|i) = τ(i→j) / Σ_k τ(i→k)  (normalized over eligible states)
   - **Sparse connectivity**: Only states with τ > 0 AND P(j|i) ≥ min_prob are connected
   - BSS thresholds filter out weak/noisy connections

**Key Benefits**:
- Clean separation of **topology traversal** vs **semantic similarity**
- BSS gives **true semantic** transitions, not just lattice adjacency
- Sparse graph: only meaningful connections (above threshold)
- Proper **conditional probability** distribution (normalized)

In [12]:
# Reload to get new 2-phase methods
import importlib
import core.hllset_transformer
importlib.reload(core.hllset_transformer)

from core.hllset_transformer import (
    HLLSetTransformer, CollectedContext, HLLMarkovChain,
    MarkovTransition, CollectedHLLSet
)

# Recreate transformer with new code
transformer = HLLSetTransformer(
    lattice=lattice,
    p_bits=P_BITS,
    tau_threshold=0.05,
    rho_threshold=1.0,
    temperature=0.7,
)

print("✓ Loaded 2-phase architecture")
print(f"  - collect_context() → CollectedContext")
print(f"  - build_markov_chain() → HLLMarkovChain")
print(f"  - forward_2phase() → (CollectedContext, HLLMarkovChain, TransformerResult)")

✓ Loaded 2-phase architecture
  - collect_context() → CollectedContext
  - build_markov_chain() → HLLMarkovChain
  - forward_2phase() → (CollectedContext, HLLMarkovChain, TransformerResult)


In [13]:
# PHASE 1: Collect HLLSets using lattice topology
print("PHASE 1: Context Collection")
print("=" * 60)

collected = transformer.collect_context(QUERY_1)

print(f"\nQuery: '{QUERY_1}'")
print(f"\nCollectedContext:")
print(f"  - Collected HLLSets: {collected.size}")
print(f"  - Total cardinality: {collected.total_cardinality:.0f} tokens")
print(f"  - Stop reason: {collected.stop_reason.value}")
print(f"  - Flux history: {[f'{f:.0f}' for f in collected.flux_history]}")

print("\n--- Collected HLLSets (with provenance) ---")
for hll_id in collected.traversal_order:
    c = collected.collected[hll_id]
    print(f"  {c.hll_id[:12]}... depth={c.depth}, t={c.level_time:.1f}, τ={c.tau_to_query:.3f}, |M|≈{c.cardinality:.0f}")

PHASE 1: Context Collection

Query: 'what is the capital of france'

CollectedContext:
  - Collected HLLSets: 5
  - Total cardinality: 172 tokens
  - Stop reason: lattice_boundary
  - Flux history: ['51', '48', '40', '26']

--- Collected HLLSets (with provenance) ---
  49cb69b90973... depth=0, t=4.0, τ=1.000, |M|≈7
  87aec656f7fe... depth=0, t=4.0, τ=0.571, |M|≈55
  189fca609cd4... depth=1, t=3.0, τ=0.714, |M|≈57
  5c0fa87d9427... depth=2, t=2.0, τ=0.714, |M|≈53
  5db5fd678b91... depth=3, t=1.0, τ=0.714, |M|≈47


In [14]:
# PHASE 2: Build Markov Chain using BSS as transition probabilities
print("PHASE 2: Build Markov Chain (BSS-based)")
print("=" * 60)

mc = transformer.build_markov_chain(collected)

print(f"\nHLLMarkovChain:")
print(f"  - States: {mc.state_count}")
print(f"  - Transitions: {mc.transition_count}")

# Show transition matrix (sparse)
print("\n--- Transition Probabilities P(j|i) ---")
matrix = mc.transition_matrix()
for from_id, row in matrix.items():
    print(f"\n  From {from_id[:10]}...:")
    for to_id, prob in sorted(row.items(), key=lambda x: -x[1])[:3]:
        print(f"    → {to_id[:10]}... P={prob:.3f}")

# Most likely path from query
print("\n--- Most Likely Path (greedy) ---")
path = mc.most_likely_path(collected.query_id, length=4)
print("  " + " → ".join(p[:10]+"..." for p in path))

PHASE 2: Build Markov Chain (BSS-based)

HLLMarkovChain:
  - States: 5
  - Transitions: 20

--- Transition Probabilities P(j|i) ---

  From 49cb69b909...:
    → 5db5fd678b... P=0.295
    → 5c0fa87d94... P=0.261
    → 189fca609c... P=0.243

  From 87aec656f7...:
    → 49cb69b909... P=0.582
    → 5db5fd678b... P=0.217
    → 189fca609c... P=0.125

  From 189fca609c...:
    → 49cb69b909... P=0.509
    → 5db5fd678b... P=0.212
    → 5c0fa87d94... P=0.188

  From 5c0fa87d94...:
    → 49cb69b909... P=0.537
    → 5db5fd678b... P=0.224
    → 189fca609c... P=0.185

  From 5db5fd678b...:
    → 49cb69b909... P=0.508
    → 5c0fa87d94... P=0.188
    → 189fca609c... P=0.175

--- Most Likely Path (greedy) ---
  49cb69b909... → 5db5fd678b... → 5c0fa87d94... → 189fca609c... → 87aec656f7...


In [15]:
# Stationary distribution - importance of each state
print("Stationary Distribution (state importance):")
print("=" * 60)

stationary = mc.stationary_distribution(iterations=50)

print("\nState importance (stationary distribution π):")
for state_id, prob in sorted(stationary.items(), key=lambda x: -x[1]):
    # Get cardinality for context
    card = mc.states[state_id].cardinality()
    print(f"  {state_id[:12]}... π={prob:.4f} (|M|≈{card:.0f} tokens)")

# The stationary distribution tells us which states are "central"
# in the semantic network - documents that connect to many others

Stationary Distribution (state importance):

State importance (stationary distribution π):
  49cb69b90973... π=0.3460 (|M|≈7 tokens)
  5db5fd678b91... π=0.2006 (|M|≈47 tokens)
  5c0fa87d9427... π=0.1684 (|M|≈53 tokens)
  189fca609cd4... π=0.1652 (|M|≈57 tokens)
  87aec656f7fe... π=0.1198 (|M|≈55 tokens)


In [16]:
# Full 2-phase forward pass (convenience method)
print("forward_2phase() - Full Pipeline")
print("=" * 60)

collected_2, mc_2, result_2 = transformer.forward_2phase(QUERY_1)

print(f"\nQuery: '{QUERY_1}'")
print(f"\nPhase 1 (Collection):")
print(f"  - HLLSets collected: {collected_2.size}")
print(f"  - Total tokens: {collected_2.total_cardinality:.0f}")

print(f"\nPhase 2 (Markov Chain):")
print(f"  - States: {mc_2.state_count}")
print(f"  - Transitions: {mc_2.transition_count}")

print(f"\nTransformerResult (compatible output):")
print(f"  - Graph nodes: {result_2.hllset_graph.node_count}")
print(f"  - Graph edges: {result_2.hllset_graph.edge_count}")
print(f"  - Convergence: {result_2.convergence_reason.value}")

print("\n✓ 2-phase architecture: clean separation of topology & semantics!")

forward_2phase() - Full Pipeline

Query: 'what is the capital of france'

Phase 1 (Collection):
  - HLLSets collected: 5
  - Total tokens: 172

Phase 2 (Markov Chain):
  - States: 5
  - Transitions: 20

TransformerResult (compatible output):
  - Graph nodes: 5
  - Graph edges: 20
  - Convergence: lattice_boundary

✓ 2-phase architecture: clean separation of topology & semantics!


---

## Part 4: Graph Disambiguation

The transformer returns a graph of HLLSets with τ-weighted edges.
Now we disambiguate to recover actual documents.

In [17]:
# For this demo, we'll manually disambiguate the final context
# (The full graph disambiguation preserves τ edges)

print("Disambiguating extracted HLLSets...")
print("=" * 60)

# Recover tokens from final context
disambiguation_results = engine.disambiguate_hllset(result_1.final_context)

# Extract best token from each result (unigrams only)
recovered_tokens = [r.best_token for r in disambiguation_results if r.best_token and ' ' not in r.best_token]

print(f"\nRecovered {len(recovered_tokens)} tokens from context:")
print(f"  {' '.join(recovered_tokens[:50])}..." if len(recovered_tokens) > 50 else f"  {' '.join(recovered_tokens)}")

Disambiguating extracted HLLSets...

Recovered 164 tokens from context:
  capitol paris transformed da modern niemeyer world architecture. a boca ottawa summer in eiffel designed london landmarks. tower. bangkok mexico. berlin states. 1789. brandenburg wat attract republic. france at olympics. big country millions hill building cuisine brazil. represents and madrid united temples began hosts contains goya aires will la by...


### Check for France-related content

In [18]:
# Look for keywords related to France
france_keywords = ['france', 'paris', 'french', 'eiffel', 'louvre', 'capital']

print("France-related tokens found:")
found = [t for t in recovered_tokens if any(kw in t.lower() for kw in france_keywords)]
print(f"  {found}")

# Verify with BSS against original France documents
france_doc = "Paris is the capital of France. It is known for the Eiffel Tower and the Louvre museum."
france_hll = HLLSet.from_batch(france_doc.lower().split(), p_bits=P_BITS)

tau_france, rho_france = bss(result_1.final_context, france_hll)
print(f"\nBSS to France document: τ={tau_france:.3f}, ρ={rho_france:.3f}")
print(f"  (τ > 0.5 indicates strong overlap)")

France-related tokens found:
  ['paris', 'eiffel', 'france', 'french', 'louvre', 'france.', 'capital']

BSS to France document: τ=1.000, ρ=1.000
  (τ > 0.5 indicates strong overlap)


---

## Part 5: Simulated LLM Synthesis

In production, we'd send the `DocumentGraph` to an external LLM.
Here, we simulate by concatenating relevant recovered content.

In [19]:
def simulate_llm_synthesis(query: str, recovered_tokens: list, original_docs: list) -> str:
    """
    Simulate LLM synthesis by finding most relevant original documents.
    
    Uses TF-IDF-like weighting: query-specific tokens count MORE than
    common words like 'the', 'is', 'of'.
    
    In production: send to GPT-4, Claude, etc.
    """
    # Query-specific keywords (weighted 10x more)
    query_tokens = set(query.lower().split())
    stopwords = {'the', 'is', 'of', 'a', 'an', 'and', 'in', 'to', 'it', 'for', 'with', 'are', 'was', 'by', 'what'}
    query_keywords = query_tokens - stopwords
    
    # Find documents that overlap with recovered tokens
    # Normalize tokens (remove punctuation for matching)
    recovered_set = set(t.lower().strip('.,!?;:') for t in recovered_tokens)
    
    scored_docs = []
    for doc in original_docs:
        doc_tokens_raw = doc.lower().split()
        doc_tokens = set(t.strip('.,!?;:') for t in doc_tokens_raw)
        
        # Base overlap score
        base_overlap = len(doc_tokens & recovered_set)
        
        # Query keyword bonus (weighted heavily)
        # Check if query keywords appear in THIS document
        doc_has_keywords = doc_tokens & query_keywords
        keyword_bonus = len(doc_has_keywords) * 20  # 20x weight for query keywords
        
        # Combined score
        score = (base_overlap + keyword_bonus) / len(doc_tokens)
        
        if score > 0.3:  # Threshold
            scored_docs.append((score, doc, doc_has_keywords))
    
    scored_docs.sort(reverse=True)
    
    # Build response
    lines = [
        f"Query: {query}",
        f"Query keywords: {query_keywords}",
        "",
        "Based on the retrieved documents:",
        "",
    ]
    
    for i, (score, doc, keywords) in enumerate(scored_docs[:3], 1):
        lines.append(f"[{i}] (score: {score:.1f}, keywords: {keywords}) {doc}")
    
    lines.append("")
    lines.append("Synthesis: " + scored_docs[0][1] if scored_docs else "No relevant documents found.")
    
    return "\n".join(lines)

# Collect all original documents
all_docs = [doc for _, _, batch in ALL_BATCHES for doc in batch]

# Simulate synthesis
synthesis_1 = simulate_llm_synthesis(QUERY_1, recovered_tokens, all_docs)

print("SIMULATED LLM RESPONSE")
print("=" * 60)
print(synthesis_1)

SIMULATED LLM RESPONSE
Query: what is the capital of france
Query keywords: {'capital', 'france'}

Based on the retrieved documents:

[1] (score: 3.9, keywords: {'capital', 'france'}) Paris is the capital of France. It is known for the Eiffel Tower and the Louvre museum.
[2] (score: 3.0, keywords: {'capital'}) Ottawa is the capital of Canada. Parliament Hill overlooks the Ottawa River.
[3] (score: 2.8, keywords: {'capital'}) Berlin is the capital of Germany. The Brandenburg Gate is a symbol of reunification.

Synthesis: Paris is the capital of France. It is known for the Eiffel Tower and the Louvre museum.


---

## Part 6: Close the Loop — Re-ingestion

The synthesized response becomes new knowledge in the lattice.
Future queries can find this synthesized content!

In [20]:
# Re-ingest the synthesis into the lattice
synthesis_tokens = synthesis_1.lower().split()
synthesis_hll = HLLSet.from_batch(synthesis_tokens, p_bits=P_BITS)

# Add to lattice at t=5.0
synthesis_node = lattice.append(
    [synthesis_hll],
    timestamp=5.0,
    metadata={
        'type': 'llm_synthesis',
        'query': QUERY_1,
        'source': 'simulated',
    }
)

# Update engine with new tokens
engine.ingest_tokens(synthesis_tokens)

print("✓ Synthesis re-ingested into W lattice")
print(f"  New level ID: {synthesis_node.node_id[:16]}...")
print(f"  Timestamp: t={synthesis_node.timestamp}")
print(f"  Cardinality: {synthesis_node.cardinality:.0f}")
print(f"\nLattice now has {lattice.node_count} levels")

✓ Synthesis re-ingested into W lattice
  New level ID: a8217fa6fb634367...
  Timestamp: t=5.0
  Cardinality: 48

Lattice now has 5 levels


---

## Part 7: Second Query — Finding the Synthesis

Now let's run another query. The transformer should find
both original documents AND our synthesized response!

In [21]:
# Engine already updated with new vocabulary in previous cell

QUERY_2 = "paris eiffel tower louvre"

print(f"Query: '{QUERY_2}'")
print("=" * 60)

result_2 = transformer.forward(QUERY_2)

print(f"\n{result_2.summary()}")
print(f"Depths traversed: {result_2.depths_traversed}")
print(f"Total information gain: {result_2.total_information_gain:.0f}")

Query: 'paris eiffel tower louvre'

Transformer: 5 levels, +118 info, lattice_boundary
Depths traversed: 5
Total information gain: 118


In [22]:
# Check if synthesis is in the context
tau_synthesis, _ = bss(result_2.final_context, synthesis_hll)
print(f"BSS to previous synthesis: τ={tau_synthesis:.3f}")

if tau_synthesis > 0.3:
    print("✓ Previous synthesis is part of the context!")
    print("  (The system remembers what it learned)")

BSS to previous synthesis: τ=1.000
✓ Previous synthesis is part of the context!
  (The system remembers what it learned)


---

## Part 8: Query About a Different Topic

In [23]:
QUERY_3 = "what is the capital of japan tokyo"

print(f"Query: '{QUERY_3}'")
print("=" * 60)

result_3 = transformer.forward(QUERY_3)

print(f"\n{result_3.summary()}")

# Recover and check
disambiguation_3 = engine.disambiguate_hllset(result_3.final_context)
recovered_3 = [r.best_token for r in disambiguation_3 if r.best_token]
japan_keywords = ['japan', 'tokyo', 'temples', 'skyscrapers', 'megacity']
found_3 = [t for t in recovered_3 if any(kw in t.lower() for kw in japan_keywords)]
print(f"\nJapan-related tokens: {found_3}")

Query: 'what is the capital of japan tokyo'

Transformer: 5 levels, +186 info, lattice_boundary

Japan-related tokens: ['temples', 'megacity', 'japan.', 'tokyo', 'skyscrapers.']


---

## Part 9: The Complete Pipeline in One Cell

Here's the entire HLLSet AI pipeline — from query to response to re-ingestion:

In [24]:
def hllset_ai_query(query: str, transformer, engine, lattice, all_docs):
    """
    Complete HLLSet AI pipeline.
    
    6 conceptual steps, production-ready.
    """
    # 1. Forward pass through transformer
    result = transformer.forward(query)
    
    # 2. Disambiguate to recover tokens
    disambiguation = engine.disambiguate_hllset(result.final_context)
    recovered = [r.best_token for r in disambiguation if r.best_token and ' ' not in r.best_token]
    
    # 3. Synthesize response (simulated LLM)
    response = simulate_llm_synthesis(query, recovered, all_docs)
    
    # 4. Re-ingest synthesis
    response_tokens = response.lower().split()
    response_hll = HLLSet.from_batch(response_tokens, p_bits=lattice.p_bits)
    new_node = lattice.append(
        [response_hll],
        metadata={'type': 'synthesis', 'query': query}
    )
    engine.ingest_tokens(response_tokens)
    
    return {
        'query': query,
        'result': result,
        'recovered_tokens': recovered,
        'response': response,
        'new_node': new_node,
    }

# Run a complete query
print("COMPLETE HLLSET AI QUERY")
print("=" * 60)

output = hllset_ai_query(
    "tell me about washington dc united states",
    transformer, engine, lattice, all_docs
)

print(f"Query: {output['query']}")
print(f"\nTransformer: {output['result'].summary()}")
print(f"\nRecovered {len(output['recovered_tokens'])} tokens")
print(f"\nResponse preview:")
print(output['response'][:300] + "...")
print(f"\n✓ Re-ingested as node {output['new_node'].node_id[:12]}...")

COMPLETE HLLSET AI QUERY
Query: tell me about washington dc united states

Transformer: Transformer: 5 levels, +88 info, lattice_boundary

Recovered 88 tokens

Response preview:
Query: tell me about washington dc united states
Query keywords: {'tell', 'united', 'washington', 'me', 'states', 'about', 'dc'}

Based on the retrieved documents:

[1] (score: 6.3, keywords: {'states', 'dc', 'united', 'washington'}) Washington DC is the capital of the United States. The White House...

✓ Re-ingested as node 621aed203aed...


---

## Summary

### What We Built

1. **Multi-layer W Lattice** — 4 batches → 4 temporal levels (+ synthesized levels)
2. **HLLSetTransformer** — Three forward pass modes:
   - `forward()`: Complement-based backward attention (merged context)
   - `forward_mc()`: W-Markov Chain with preserved provenance (legacy)
   - `forward_2phase()`: **NEW 2-phase architecture** (recommended)
3. **Graph Disambiguation** — Token recovery with relationship preservation
4. **Simulated LLM Synthesis** — TF-IDF weighted document matching
5. **Closed Loop** — Synthesis re-ingested, future queries find it

### The 2-Phase Architecture (NEW!)

**Clean separation of concerns:**

```
┌─────────────────────────────────────────────────────────────────┐
│  PHASE 1: collect_context()                                     │
│  ─────────────────────────────────────────────────────────────  │
│  Uses LATTICE TOPOLOGY to collect relevant HLLSets              │
│                                                                 │
│  Query → Traverse W(t) → Check τ(query→level) → Collect IDs     │
│                                                                 │
│  Output: CollectedContext {                                     │
│      collected: {id₁: HLLSet₁, id₂: HLLSet₂, ...}               │
│      traversal_order: [id₁, id₂, ...]                           │
│      stop_reason: noether_converged | max_depth | boundary      │
│  }                                                              │
└─────────────────────────────────────────────────────────────────┘
                              │
                              ▼
┌─────────────────────────────────────────────────────────────────┐
│  PHASE 2: build_markov_chain()                                  │
│  ─────────────────────────────────────────────────────────────  │
│  Uses BSS for SEMANTIC TRANSITION PROBABILITIES                 │
│                                                                 │
│  P(j|i) = τ(i→j) / Σ_k τ(i→k)  where τ(i→k) > 0                 │
│  Only edges with P(j|i) ≥ min_prob are included (sparse graph)  │
│                                                                 │
│  Output: HLLMarkovChain {                                       │
│      states: {id → HLLSet}                                      │
│      transitions: {from_id → [(to_id, P, τ, ρ), ...]}           │
│      stationary_distribution(): {id → π}  (importance)          │
│      most_likely_path(): [id₁, id₂, ...]                        │
│  }                                                              │
└─────────────────────────────────────────────────────────────────┘
```

**Key Benefits:**
- **Separation of concerns**: Topology traversal ≠ semantic similarity
- **Sparse connectivity**: Only states with τ > 0 AND P ≥ threshold connected
- **Proper probability**: P(j|i) normalized over eligible transitions
- **Stationary distribution**: Find "central" documents in semantic space
- **Reusable collection**: Compute multiple MC views from same collection

### The Recommended Pipeline

```python
# One-call 2-phase forward
collected, mc, result = transformer.forward_2phase("your query")

# Phase 1 output: individual HLLSets with provenance
print(f"Collected {collected.size} HLLSets")
for hll_id, c in collected.collected.items():
    print(f"  {hll_id[:12]}... depth={c.depth}, τ={c.tau_to_query:.3f}")

# Phase 2 output: Markov Chain with semantic transitions
path = mc.most_likely_path(collected.query_id, length=3)
importance = mc.stationary_distribution()

# Downstream: disambiguation, synthesis, re-ingestion
recovered = engine.disambiguate_hllset(result.final_context)
response = llm.synthesize(recovered)
lattice.append([HLLSet.from_batch(response)])
```

### Key Properties

| Property | HLLSet AI | Traditional LLM |
|----------|-----------|----------------|
| **Storage** | ~640 bytes/document | GB of parameters |
| **Retrieval** | BSS τ (O(1) per level) | Embedding + ANN |
| **Provenance** | Full trace (SHA1 IDs) | Opaque |
| **Learning** | Re-ingestion (instant) | Fine-tuning (expensive) |
| **Hardware** | Any CPU | GPU cluster |
| **Transitions** | BSS-based P(j\|i) | N/A |
| **Importance** | Stationary dist. π | N/A |

### What the External LLM Does

Only one thing: **creative synthesis**. The HLLSet pipeline handles:
- Retrieval ✓
- Context assembly ✓  
- Relationship tracking ✓
- **Provenance (SHA1 IDs)** ✓
- **Markov transitions** ✓
- **State importance (π)** ✓
- Knowledge accumulation ✓

The LLM turns structured context into fluent text. That's all.

---

**The HLLSet AI model: 2-phase architecture, clean separation, runs on a phone.**

In [ ]:
# Final statistics
print("FINAL SYSTEM STATE")
print("=" * 60)
print(f"W Lattice levels: {lattice.node_count}")
print(f"Total corpus coverage: {lattice.cumulative().cardinality():.0f} unique tokens")
print(f"\nLevel history:")
for node in lattice.all_nodes():
    meta = node.metadata.get('type', node.metadata.get('batch_name', 'original'))
    print(f"  t={node.timestamp}: {meta} (|M|≈{node.cardinality:.0f})")